# KMeans - Customer Segmentation

This notebook is designed for classroom demonstration with clear comments and simple steps.

## Objective

In this notebook, we will demonstrate a basic **clustering** machine learning flow.

## Basic Steps

1. Read the dataset  
2. Perform basic EDA  
3. Prepare input features and target  
4. Convert categorical columns into numeric columns  
5. Split data into training and testing sets  
6. Train the model  
7. Predict  
8. Evaluate using suitable metrics  
9. Save the trained model using pickle  

Dataset used: `customer_segmentation_1000.csv`


In [1]:
# ============================================================
# Import required libraries
# ============================================================

import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Read dataset
df = pd.read_csv("../datasets/customer_segmentation_1000.csv")
df.head()

,annual_income,spending_score,website_visits,app_sessions,avg_order_value,orders_per_month,return_rate,loyalty_years,region,membership
0,2377900,60,17,25,7167.19,11,0.13,4.6,North,Silver
1,1328435,77,15,19,12314.62,6,0.16,6.5,North,Silver
2,1572299,82,18,23,3477.19,6,0.20,3.8,South,Platinum
3,1105309,56,15,20,4885.06,6,0.23,4.4,South,Gold
4,272884,9,10,15,6754.48,9,0.19,0.3,East,Silver


In [3]:
# Basic EDA
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())

df.describe()

Dataset shape: (1000, 10)

Column names:
['annual_income', 'spending_score', 'website_visits', 'app_sessions', 'avg_order_value', 'orders_per_month', 'return_rate', 'loyalty_years', 'region', 'membership']

Missing values:
annual_income       0
spending_score      0
website_visits      0
app_sessions        0
avg_order_value     0
orders_per_month    0
return_rate         0
loyalty_years       0
region              0
membership          0
dtype: int64


,annual_income,spending_score,website_visits,app_sessions,avg_order_value,orders_per_month,return_rate,loyalty_years
count,1.000000e+03,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000
mean,1.335486e+06,49.207000,12.175000,20.23100,7621.072740,4.952000,0.196350,5.873100
std,6.878873e+05,28.607141,3.579551,4.44583,4144.961912,2.207843,0.115276,3.444577
min,2.006170e+05,1.000000,3.000000,6.00000,322.890000,0.000000,0.000000,0.000000
25%,7.033290e+05,24.000000,10.000000,17.00000,4207.167500,3.000000,0.100000,2.900000
50%,1.363144e+06,50.000000,12.000000,20.00000,7520.460000,5.000000,0.190000,5.800000
75%,1.946781e+06,74.000000,14.000000,23.00000,11163.225000,6.000000,0.290000,8.825000
max,2.497496e+06,100.000000,25.000000,36.00000,14999.620000,12.000000,0.400000,12.000000


In [4]:
# Prepare data for clustering

# KMeans is unsupervised learning.
# So there is no target column like churn or price.

X = pd.get_dummies(df, drop_first=True)

print("Input shape after encoding:", X.shape)
X.head()

Input shape after encoding: (1000, 13)


,annual_income,spending_score,website_visits,app_sessions,avg_order_value,orders_per_month,return_rate,loyalty_years,region_North,region_South,region_West,membership_Platinum,membership_Silver
0,2377900,60,17,25,7167.19,11,0.13,4.6,True,False,False,False,True
1,1328435,77,15,19,12314.62,6,0.16,6.5,True,False,False,False,True
2,1572299,82,18,23,3477.19,6,0.20,3.8,False,True,False,True,False
3,1105309,56,15,20,4885.06,6,0.23,4.4,False,True,False,False,False
4,272884,9,10,15,6754.48,9,0.19,0.3,False,False,False,False,True


In [5]:
# Scale data

# KMeans uses distance.
# Scaling is important so large-value columns do not dominate small-value columns.

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaling completed.")

Scaling completed.


In [6]:
# Train KMeans model

# n_clusters=4 means we want to group customers into 4 segments.
model = KMeans(n_clusters=4, random_state=42, n_init=10)

clusters = model.fit_predict(X_scaled)

df["cluster"] = clusters

df.head()

,annual_income,spending_score,website_visits,app_sessions,avg_order_value,orders_per_month,return_rate,loyalty_years,region,membership,cluster
0,2377900,60,17,25,7167.19,11,0.13,4.6,North,Silver,2
1,1328435,77,15,19,12314.62,6,0.16,6.5,North,Silver,2
2,1572299,82,18,23,3477.19,6,0.20,3.8,South,Platinum,0
3,1105309,56,15,20,4885.06,6,0.23,4.4,South,Gold,1
4,272884,9,10,15,6754.48,9,0.19,0.3,East,Silver,1


In [7]:
# ============================================================
# Evaluation metrics for clustering
# ============================================================

# In clustering, we do not have actual labels.
# So we use unsupervised metrics.

# Inertia:
# Measures how close records are to their assigned cluster center.
# Lower inertia is generally better, but too many clusters can reduce inertia.
inertia = model.inertia_

# Silhouette Score:
# Measures how well separated the clusters are.
# Range is approximately -1 to 1.
# Higher value means better cluster separation.
silhouette = silhouette_score(X_scaled, clusters)

# Davies-Bouldin Score:
# Measures average similarity between clusters.
# Lower value is better.
db_score = davies_bouldin_score(X_scaled, clusters)

print("Inertia             :", round(inertia, 2))
print("Silhouette Score    :", round(silhouette, 4))
print("Davies-Bouldin Score:", round(db_score, 4))

Inertia             : 9823.78
Silhouette Score    : 0.1393
Davies-Bouldin Score: 2.27


In [8]:
# Save clustering output
df.to_csv("../datasets/customer_segmentation_with_clusters.csv", index=False)

print("Clustered dataset saved successfully.")

Clustered dataset saved successfully.


In [9]:
# Save scaler and KMeans model using pickle

kmeans_package = {
    "scaler": scaler,
    "model": model
}

model_path = "../models/kmeans_customer_segmentation.pkl"

with open(model_path, "wb") as file:
    pickle.dump(kmeans_package, file)

print("KMeans model and scaler saved successfully at:", model_path)

KMeans model and scaler saved successfully at: ../models/kmeans_customer_segmentation.pkl
